# Enfoque con Embeddings
Todo a embeddings (queries, sinopsis + año + director + keywords) --> 5 con mas similtud coseno (comparando queries vs sinopsis + año + director + keywords)

La estrategia consiste en representar tanto las películas como las preferencias de cada usuario en un espacio vectorial común, y recomendar las películas cuyo vector sea más similar al perfil del usuario.

1. unificar texto
2. embeddings con w2v o sentence transformer sobre texto y queries
3. similitud coseno text vs queries

In [1]:
!pip install datasets
!pip install sentence-transformers
!pip install gensim


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached gensim-4.4.0.tar.gz (23.3 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached smart_open-7.6.1-py3-none-any.whl.metadata (25 kB)
  Using cached wrapt-2.2.1-cp314-cp314-win_amd64.whl.metadata (7.6 kB)
Using cached smart_open-7.6.1-py3-none-any.whl (64 kB)
Using cached wrapt-2.2.1-cp314-cp314-win_amd64.whl (81 kB)
Failed to build gensim


  error: subprocess-exited-with-error
  
  × Building wheel for gensim (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [716 lines of output]
      C:\Users\recla\AppData\Local\Temp\pip-build-env-t89oqbnu\overlay\Lib\site-packages\setuptools\_distutils\dist.py:287: UserWarning: Unknown distribution option: 'test_suite'
        warnings.warn(msg)
      C:\Users\recla\AppData\Local\Temp\pip-build-env-t89oqbnu\overlay\Lib\site-packages\setuptools\_distutils\dist.py:287: UserWarning: Unknown distribution option: 'tests_require'
        warnings.warn(msg)
      running bdist_wheel
      running build
      running build_py
      creating build\lib.win-amd64-cpython-314\gensim
      copying gensim\downloader.py -> build\lib.win-amd64-cpython-314\gensim
      copying gensim\interfaces.py -> build\lib.win-amd64-cpython-314\gensim
      copying gensim\matutils.py -> build\lib.win-amd64-cpython-314\gensim
      copying gensim\nosy.py -> build\lib.win-amd64-cpython-314\gensim
   

In [2]:
import pandas as pd
import re
import numpy as np
from collections import Counter

# mdoelo
from sentence_transformers import SentenceTransformer


from sklearn.metrics.pairwise import cosine_similarity

c:\Users\recla\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Carga de datasets

Traemos el dataset de sinopsis de peliculas de IMDb desde Hugging Face

In [3]:
df_pelis = pd.read_csv("https://raw.githubusercontent.com/nazarenomm/Sistema-de-recomendacion-de-peliculas/refs/heads/main/data/peliculas_limpio.csv")

Traemos el dataset proporcionado con la información de los usuarios y sus respectivas queries

In [4]:
usuarios = pd.read_csv("https://raw.githubusercontent.com/nazarenomm/Sistema-de-recomendacion-de-peliculas/refs/heads/main/data/usuarios.csv")

Visualizamos las queries

In [5]:
for texto in usuarios['query']:
    print(texto)

Quiero una película donde una mujer enfrenta una amenaza invisible que viene de alguien cercano
Busco algo basado en hechos reales sobre corrupción o poder político
Una comedia donde la relación entre dos personas empieza de forma ridícula o accidental
Algo que haga pensar sobre qué es real y qué es una construcción, con acción pero también ideas
Animación donde el protagonista lucha por su libertad o identidad en un mundo que lo oprime
Un grupo de personas planea un robo o estafa y las cosas se complican de forma inesperada
Una película sobre músicos o artistas que viven al margen, con mucha atmósfera y estilo visual
Acción directa con un héroe que trabaja solo o casi solo contra una organización criminal o corrupta
Algo tranquilo sobre personas que intentan reconectar o entenderse después de una distancia larga
Algo que sea difícil de clasificar, con una lógica narrativa propia, no convencional
No sé bien, algo que valga la pena ver un domingo a la noche, que enganche desde el princi

Construimos el dataset de peliculas, agregando un id que faltaba.

In [6]:
df_pelis["id"] = df_pelis.index + 1
df_pelis.head()

,description,keywords,genre,year,name,director,id
0,"Orin Boyd, un duro policía de una comisaría de...","vietnam war veteran, heroína, drogas, narcotra...","acción, crimen, suspense",2001.0,Herida abierta,Andrzej Bartkowiak,1
1,Al llegar a un pequeño pueblo donde ha heredad...,"herencia, hostess, comedia negra, pueblo, magia","comedia, terror",1989.0,"Elvira, reina de las tinieblas",James Signorelli,2
2,Una mujer finge su muerte en un intento de esc...,"violencia doméstica, muerte fingida, borderlin...","drama, suspense",1991.0,Durmiendo con su enemigo,Joseph Ruben,3
3,Durante un memorial en la ciudad natal de su p...,"manic pixie dream girl, publicidad, bad public...","comedia, drama, romance",2005.0,Elizabethtown,Cameron Crowe,4
4,Las pruebas nucleares francesas irradian a una...,"monstruo gigante, iguana, militar, giant footp...","acción, ciencia ficción, suspense",1998.0,Godzilla,Roland Emmerich,5


## Preprocesado

### Función de limpieza de texto

Aplicamos un pipeline de limpieza estándar: minúsculas, eliminación de puntuación y palabras con números. 


In [7]:
def limpiar_texto(text):
    if not isinstance(text, str):
        return ''
    text = re.sub(r'<[^>]+>', ' ', text)        # remover HTML si hubiera
    text = re.sub(r'[^\w\s\.,;:!?áéíóúüñ-]', ' ', text)  # caracteres extraños
    text = re.sub(r'\s+', ' ', text)             # espacios múltiples
    return text.strip()

Unificamos las variables relevantes en un texto (todas menos año)

In [8]:
df_pelis["texto"] = (
    df_pelis["name"].apply(limpiar_texto) + ". "
    + df_pelis["description"].apply(limpiar_texto) + " "
    + df_pelis["director"].fillna('').apply(limpiar_texto) + ". " # probar agregar queries como "quiero ver una de Tarantino" a ver si funciona, sino sacar
    + df_pelis["year"].apply(lambda x: str(int(x)) if pd.notnull(x) else '') + ". "
    + df_pelis["genre"].apply(limpiar_texto) + ". "
    + df_pelis["keywords"].apply(limpiar_texto)
)

In [9]:
df_pelis.texto.iloc[0]

'Herida abierta. Orin Boyd, un duro policía de una comisaría del centro de la ciudad, descubre una red de policías corruptos. Andrzej Bartkowiak. 2001. acción, crimen, suspense. vietnam war veteran, heroína, drogas, narcotraficante, corrupt cop'

#### El espanglish en ``genre`` y ``keywords``:  
El modelo multilingüe maneja texto en múltiples idiomas, pero fue entrenado con documentos monolingües por separado, no necesariamente con mezcla de idiomas dentro del mismo string

¿Es un problema grave? No. El modelo multilingüe es bastante robusto a esto. Pero sí es algo valioso para mencionar como limitación del corpus.

## Embedding de peliculas

In [10]:
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4681.09it/s]


### Calcular embedding de cada película

No hace falta hacer el promedio para calcularlo, el output del modelo ya es el embedding del documento (pelicula).

In [11]:
pelis_embeddings = model.encode(df_pelis['texto'].tolist(), show_progress_bar=True)

Batches:   0%|          | 0/156 [00:00<?, ?it/s]

Batches: 100%|██████████| 156/156 [02:39<00:00,  1.02s/it]


Como el dataset de películas y el de embeddings se construyeron en el mismo orden,podemos unirlos directamente por índice

In [12]:
df_pelis = df_pelis.reset_index(drop=True)

pelis_embeddings_df = pd.DataFrame(pelis_embeddings)
pelis_embeddings_df = pelis_embeddings_df.merge(
    df_pelis[['id', 'name']], 
    left_index=True, 
    right_index=True
)

In [13]:
pelis_embeddings_df.head()

,0,1,2,3,4,5,6,7,8,9,...,376,377,378,379,380,381,382,383,id,name
0,-0.031083,0.032719,-0.272750,0.105669,0.088466,0.341037,0.109469,0.170736,0.048047,-0.134747,...,-0.053232,0.032655,-0.199530,-0.102709,0.034748,-0.105462,0.243188,0.014966,1,Herida abierta
1,-0.007789,0.076888,-0.077062,0.274105,-0.079885,-0.025508,0.314847,0.059361,0.081432,0.050591,...,0.168026,0.220522,-0.022279,-0.104578,0.325791,-0.082222,-0.040965,-0.080095,2,"Elvira, reina de las tinieblas"
2,-0.017430,0.006371,-0.100607,0.260501,0.053605,0.227207,0.040888,-0.065317,0.184578,-0.041667,...,0.132501,0.318629,-0.022673,0.075589,0.229808,0.034454,0.114800,-0.044366,3,Durmiendo con su enemigo
3,0.137272,0.015918,0.013975,0.060339,-0.133089,0.079739,0.129728,-0.093552,0.063227,0.041110,...,-0.029802,0.355402,0.058575,-0.094542,0.238030,0.101194,0.021751,-0.007258,4,Elizabethtown
4,-0.165777,0.050897,-0.000996,-0.207756,0.106295,-0.271873,-0.016602,0.089710,0.047141,0.133440,...,-0.091462,-0.023140,-0.099317,0.237066,0.205077,-0.302631,0.368318,0.098505,5,Godzilla


## Embeddings de usuarios

Aplicamos la misma función de limpieza que usamos para las sinopsis. La verdad que no hace falta pero debería ser parte del pipeline operativo habitual.

In [14]:
# data_clean_users = pd.DataFrame(usuarios["query"].apply(limpiar_texto))

### Sin promediar

Con ``sentence_transformer`` no hace falta calcular los promedios de la query y el historial, se puede pasar todo el texto de una.

La desventaja es que no podemos controlar la importancia que se le da al historial

el modelo tiene limite de 512 tokens, podemos chequear antes de calcular los embeddings.

In [15]:
# contar cantidad de palabras en df["texto"]
df_pelis["word_count"] = df_pelis["texto"].apply(lambda x: len(str(x).split()))
print("Cantidad maxima de palabras:")
print(df_pelis["word_count"].max())
print("Mediana de palabras:")
print(df_pelis["word_count"].median())

Cantidad maxima de palabras:
79
Mediana de palabras:
45.0


In [16]:
usuarios["query_word_count"] = usuarios["query"].apply(lambda x: len(str(x).split()))
print("Cantidad maxima de palabras en queries de usuarios:")
print(usuarios["query_word_count"].max())
print("Mediana de palabras en queries de usuarios:")
print(usuarios["query_word_count"].median())

Cantidad maxima de palabras en queries de usuarios:
19
Mediana de palabras en queries de usuarios:
15.5


Igualmente hay que considerar que las queries son cortas, podrían haber más largas en un futuro (quizás plantear una longitud máxima)

En promedio una palabra española se tokeniza en 1.5-2 tokens. Entonces:

``44 palabras × 5 películas + query (max 30 palabras, por ejemplo) ≈ 250 palabras ≈ 375-500 tokens``

Ejemplo exagerado del tokenizador

In [17]:
tokenizer = model.tokenizer

texto = "narcotraficante corruptísimo anticonstitucional"
tokens = tokenizer.tokenize(texto)
print(tokens)
print(f"palabras: 3 → tokens: {len(tokens)}")

['▁na', 'rc', 'otra', 'fica', 'nte', '▁', 'corrupt', 'ísimo', '▁antico', 'n', 'stitu', 'cional']
palabras: 3 → tokens: 12


In [18]:
def build_user_text(row, df_pelis):
    # Query del usuario
    partes = [row['query']]
    
    # Descripciones de las 5 películas del historial
    for col in ['pelicula_1','pelicula_2','pelicula_3','pelicula_4','pelicula_5']:
        nombre = row[col]
        match = df_pelis[df_pelis['name'] == nombre]['texto']
        if len(match):
            partes.append(match.values[0])
    
    return ' '.join(partes)

user_texts = usuarios.apply(lambda r: build_user_text(r, df_pelis), axis=1).tolist()

In [19]:
user_embeddings = model.encode(user_texts, show_progress_bar=True)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  1.14it/s]


In [20]:
user_embeddings_df = pd.DataFrame(user_embeddings)
user_embeddings_df = user_embeddings_df.merge(usuarios[['id']], left_index=True, right_index=True)
user_embeddings_df.head()

,0,1,2,3,4,5,6,7,8,9,...,375,376,377,378,379,380,381,382,383,id
0,-0.238107,-0.128500,-0.207465,0.260500,0.247655,0.290276,0.095238,-0.093639,0.200505,0.013914,...,0.297109,0.074182,0.340460,-0.116549,0.221071,0.159515,-0.142231,0.079386,-0.026740,U01
1,-0.249083,0.191250,-0.289829,0.216814,0.266727,0.115557,0.088643,-0.020128,0.025625,0.020821,...,-0.078047,-0.033263,0.171906,-0.238212,-0.107696,0.169403,-0.052990,-0.165263,-0.162586,U02
2,-0.010083,-0.286078,-0.028388,-0.073591,0.070636,0.379458,0.180789,0.056127,0.217427,-0.043305,...,0.053136,0.069376,0.004023,-0.328375,-0.023175,0.125202,0.085754,0.105665,0.007408,U03
3,-0.135121,-0.022123,-0.174233,-0.013320,0.024583,-0.171468,0.109334,0.055659,0.042445,0.104168,...,-0.050089,-0.095498,0.186531,-0.198212,0.121175,0.162361,0.030514,-0.036455,0.011859,U04
4,-0.022891,0.008990,-0.241990,0.069403,0.142337,0.157824,0.238181,-0.189650,0.251596,0.146036,...,0.137518,0.138678,0.324707,0.010308,0.202955,0.194683,-0.085762,-0.002339,-0.013053,U05


### Promediando

In [21]:
queries_embeddings = model.encode(usuarios['query'].tolist(), show_progress_bar=True)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  4.99it/s]


In [22]:
def get_all_historial_embeddings(usuarios_df, pelis_embeddings_df):
    all_embeddings = []
    
    for _, usuario_row in usuarios_df.iterrows():
        historial_embeddings = []
        
        for col in ['pelicula_1','pelicula_2','pelicula_3','pelicula_4','pelicula_5']:
            nombre = usuario_row[col]
            peli_emb = pelis_embeddings_df[pelis_embeddings_df['name'] == nombre]
            if not peli_emb.empty:
                historial_embeddings.append(peli_emb.drop(columns=['id', 'name']).values[0])
            else:
                print(f"Película '{nombre}' no encontrada.")
        
        embedding_promedio = np.mean(historial_embeddings, axis=0)
        all_embeddings.append(embedding_promedio)
    
    return np.array(all_embeddings)

historial_embeddings = get_all_historial_embeddings(usuarios, pelis_embeddings_df)

Película 'Rec' no encontrada.
Película 'El secreto de sus ojos' no encontrada.
Película 'El exorcista' no encontrada.
Película 'Intocable' no encontrada.
Película 'Una mente brillante' no encontrada.
Película 'Paddington' no encontrada.


In [23]:
!sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip install ollama

"sudo" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.
"sh" no se reconoce como un comando interno o externo,
programa o archivo por lotes ejecutable.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### Configuración de Ollama

Iniciamos el servidor de Ollama en background y descargamos el modelo a usar.

In [24]:
import subprocess, time, ollama

# Iniciar servidor Ollama en background
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)  # esperar que levante

# Descargar modelo (solo la primera vez)
ollama.pull('llama3.2')

FileNotFoundError: [WinError 2] El sistema no puede encontrar el archivo especificado

### Reescritura de queries con LLM

Antes de calcular embeddings, usamos el LLM para reescribir la query del usuario como si fuera una sinopsis cinematográfica. Esto mejora la similitud semántica con las descripciones de las películas en el dataset, ya que el espacio vectorial fue construido con sinopsis, no con frases coloquiales.

**¿Por qué funciona?** El modelo de embeddings proyecta textos en un espacio donde la cercanía refleja similitud semántica. Una query como *'quiero algo de suspenso psicológico'* y una sinopsis como *'un thriller psicológico donde...'* quedarán más cerca si la query se reescribe en el mismo registro que las sinopsis.

In [ ]:
def reescribir_query(query: str) -> str:
    """
    Usa el LLM para reescribir la query del usuario como una sinopsis cinematográfica.
    Esto mejora la compatibilidad semántica con las descripciones del dataset.
    """
    prompt = f"""You are a film specialist. Your task is to rewrite a user's request as if it were the beginning of a film synopsis, using a narrative and descriptive style.
    Do not add titles, film names, or explanations. Just return the rewritten text in spanish.
    User request: \"{query}\"

    rewritten synopsis:"""

    response = ollama.chat(
        model='llama3.2',
        messages=[{'role': 'user', 'content': prompt}]
    )
    return response['message']['content'].strip()


# Reescribir todas las queries
queries_reescritas = []
for query in usuarios['query']:
    reescrita = reescribir_query(query)
    queries_reescritas.append(reescrita)
    print(f"Original:   {query}")
    print(f"Reescrita:  {reescrita}")
    print()

usuarios['query_reescrita'] = queries_reescritas


### Inferencia dinámica de pesos y dirección con LLM

En lugar de usar pesos fijos `[0.7, 0.3]`, pedimos al LLM que infiera dos parámetros por usuario a partir de su query:

- **`weights`**: cuánto peso darle a la query vs. el historial en `np.average`
- **`direccion`**: si buscar películas *similares* al perfil o *distintas* (útil cuando el usuario pide explorar géneros nuevos o salir de su zona de confort)

Esto hace que el pipeline se adapte semánticamente a la intención del usuario en lugar de asumir siempre el mismo comportamiento.

In [ ]:
import json as json_lib

def inferir_parametros_llm(query: str) -> dict:
    """
    Usa el LLM para inferir dinámicamente los pesos query/historial
    y la dirección de búsqueda (similar o distinto) a partir de la query.

    Retorna un dict con:
      - 'weights': [w_query, w_historial] (suman 1)
      - 'direccion': 'similar' o 'distinto'
    """
    prompt = f"""You are a movie recommendation system. Your task is to analyze an user's request and determine two parameters to personalize the search.

**Parameters to infer:**

1. `weights`: a list [query_weight, history_weight] that must add up to exactly 1.0.

- Increase the query weight when the user knows exactly what they want (mention genre, director, specific theme, particular mood).

- Increase the history weight when the query is vague or generic ('I want something good', 'recommend something').

2. `direction`: indicates whether to recommend movies similar to or different from the user's profile.

- Use 'similar' when the user wants more of the same (their favorite genre, same style, continue watching something similar to what they've already seen).

- Use 'different' when the user asks to step outside their comfort zone, try something new, something different from the usual, or mentions wanting a change.

User query: \"{query}\"

The response will be valid only with the following JSON format, without the explanation or additional text:
{{\n
    \"weights\": [<w_query>, <w_historial>],\n
    \"direccion\": \"similar\" o \"distinct\"\n
}}"""

    response = ollama.chat(
        model='llama3.2',
        messages=[{'role': 'user', 'content': prompt}],
        format='json'  # fuerza respuesta JSON
    )

    raw = response['message']['content'].strip()

    try:
        parsed = json_lib.loads(raw)
        # Validaciones básicas
        weights = parsed.get('weights', [0.7, 0.3])
        direccion = parsed.get('direccion', 'similar')
        assert len(weights) == 2, "weights debe tener exactamente 2 elementos"
        assert abs(sum(weights) - 1.0) < 0.01, f"weights deben sumar 1, suman {sum(weights)}"
        assert direccion in ('similar', 'distinto'), f"direccion inválida: {direccion}"
        return {'weights': weights, 'direccion': direccion}
    except Exception as e:
        print(f"[WARN] Error parseando respuesta LLM: {e}. Usando defaults.")
        return {'weights': [0.7, 0.3], 'direccion': 'similar'}


# Inferir parámetros para cada usuario
parametros_usuarios = []
for query in usuarios['query']:
    params = inferir_parametros_llm(query)
    parametros_usuarios.append(params)
    print(f"Query: {query}")
    print(f"  → weights={params['weights']}, direccion='{params['direccion']}'")
    print()


### Construcción del embedding de usuario con pesos dinámicos

Usamos las queries **reescritas** (más parecidas a sinopsis) para calcular el embedding de la query, y los pesos inferidos por el LLM para ponderar query vs. historial.

In [ ]:
# Embeddings de las queries REESCRITAS (mejor alineadas con el espacio de sinopsis)
queries_reescritas_embeddings = model.encode(usuarios['query_reescrita'].tolist(), show_progress_bar=True)

# Construir embedding por usuario con pesos dinámicos
user_embeddings_dinamico = []

for i, params in enumerate(parametros_usuarios):
    w = params['weights']
    emb = np.average(
        [queries_reescritas_embeddings[i], historial_embeddings[i]],
        axis=0,
        weights=w
    )
    user_embeddings_dinamico.append(emb)

user_embeddings_dinamico = np.array(user_embeddings_dinamico)


## Recomendaciones

### Dinámico (LLM)

In [ ]:
scores_dinamico = cosine_similarity(user_embeddings_dinamico, pelis_embeddings)
# TODO: filtrar películas ya vistas en el historial

# Top-5 con dirección dinámica por usuario
top5_indices_dinamico = []
for i, params in enumerate(parametros_usuarios):
    if params['direccion'] == 'distinto':
        indices = scores_dinamico[i].argsort()[:5]          # bottom-5: más lejanas
    else:
        indices = scores_dinamico[i].argsort()[-5:][::-1]   # top-5: más cercanas
    top5_indices_dinamico.append(indices)

# Ver resultados
for i, row in usuarios.iterrows():
    params = parametros_usuarios[i]
    print(f"\n{row['nombre']} ({row['tipo_perfil']})")
    print(f"Query:      {row['query']}")
    print(f"Reescrita:  {row['query_reescrita']}")
    print(f"Pesos:      query={params['weights'][0]}, historial={params['weights'][1]}")
    print(f"Dirección:  {params['direccion']}")
    for idx in top5_indices_dinamico[i]:
        pelicula = df_pelis.iloc[idx]
        score = scores_dinamico[i, idx]
        print(f"  {pelicula['name']} ({int(pelicula['year']) if not pd.isna(pelicula['year']) else 'N/A'}) — {score:.4f}")


## Validación

In [ ]:
contador_generos = Counter()

for generos_str in df_pelis['genre']:
    generos_str = generos_str.strip('[]')
    generos_list = [g.strip() for g in generos_str.split(',')]
    contador_generos.update(generos_list)

# convertir a DataFrame
df_generos = pd.DataFrame(
    list(contador_generos.items()),
    columns=['Género', 'Frecuencia']
).sort_values('Frecuencia', ascending=False).reset_index(drop=True)

print(df_generos)

In [ ]:
resultados = []

for idx, usuario in usuarios.iterrows():
    usuario_id = usuario['id']
    generos_usuario = Counter()
    
    for col in ['pelicula_1', 'pelicula_2', 'pelicula_3', 'pelicula_4', 'pelicula_5']:
        nombre_pelicula = usuario[col]
        pelicula = df_pelis[df_pelis['name'] == nombre_pelicula]
        
        if not pelicula.empty:
            generos_str = pelicula.iloc[0]['genre']
            generos_str = generos_str.strip('[]')
            generos_list = [g.strip() for g in generos_str.split(',')]
            generos_usuario.update(generos_list)
    
    # obtener top 3-4 géneros por usuario
    top_generos = generos_usuario.most_common(5)
    generos_str = ', '.join([f"{g} ({f})" for g, f in top_generos])
    
    resultados.append({
        'Usuario': usuario_id,
        'Géneros historial': generos_str
    })

conteo_hist = pd.DataFrame(resultados)
print(conteo_hist.to_string())

In [ ]:
for i in range(len(usuarios)):
    print(f"U{i+1}: " + usuarios["query"].iloc[i])

In [ ]:
etiquetas_a_ojo_def = [
    ["suspense", "terror", "drama"],
    ["crimen", "biografía", "historia"],
    ["comedia", "romance", "drama"],
    ["acción", "ciencia ficción", "suspense"],
    ["animación", "drama", "aventura"],
    ["crimen", "acción", "comedia"],
    ["música", "drama", "comedia"],
    ["acción", "crimen", "aventura"],
    ["drama", "romance", "comedia"],
    # el U10 es ambiguo, la etiqueta debe estar mal
]

### Dinámico (LLM)

Evaluamos el enfoque con pesos dinámicos y dirección inferida. Para usuarios con `direccion='distinto'`, los géneros esperados se invierten (queremos que las recomendaciones sean distintas al historial), por lo que la métrica de precisión se interpreta diferente.

In [ ]:
resultados_eval_din = []

for user_idx, row in usuarios.head(9).iterrows():
    print(f"\n{'='*70}")
    print(f"{row['nombre']} ({row['tipo_perfil']})")
    print(f"Query: {row['query']}")
    params = parametros_usuarios[user_idx]
    print(f"Pesos: {params['weights']} | Dirección: {params['direccion']}")
    generos_esperados = set(etiquetas_a_ojo_def[user_idx])
    print(f"Géneros Esperados: {', '.join(generos_esperados)}")
    print(f"{'='*70}")

    generos_recomendados = Counter()
    peliculas_buenas = 0
    peliculas_malas = []

    print("\nTop-5 Recomendaciones:")
    for rank, idx in enumerate(top5_indices_dinamico[user_idx], 1):
        pelicula = df_pelis.iloc[idx]
        score = scores_dinamico[user_idx, idx]

        generos_str = pelicula['genre'].strip('[]')
        generos_list = [g.strip() for g in generos_str.split(',')]
        generos_pelicula = set(generos_list)
        generos_recomendados.update(generos_list)

        es_buena = bool(generos_pelicula & generos_esperados)
        if es_buena:
            peliculas_buenas += 1
            marker = "✓"
        else:
            peliculas_malas.append(pelicula['name'])
            marker = "✗"

        print(f"  {rank}. [{marker}] {pelicula['name']} ({int(pelicula['year']) if not pd.isna(pelicula['year']) else 'N/A'}) — {score:.4f}")
        print(f"     Géneros: {', '.join(generos_list)}")

    print(f"\nConteo de Géneros en Recomendaciones:")
    for genero, freq in generos_recomendados.most_common():
        print(f"  {genero}: {freq}")

    generos_capturados = set(generos_recomendados.keys())
    recall = len(generos_capturados & generos_esperados) / len(generos_esperados)
    precision = peliculas_buenas / 5
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    print(f"\nMÉTRICAS:")
    print(f"  Recall (géneros):      {recall:.1%}  ({len(generos_capturados & generos_esperados)}/{len(generos_esperados)})")
    print(f"  Precision (películas): {precision:.1%}  ({peliculas_buenas}/5)")
    print(f"  F1-Score:              {f1:.1%}")

    if peliculas_malas:
        print(f"\nPelículas problemáticas (sin géneros esperados):")
        for pelicula in peliculas_malas:
            print(f"    - {pelicula}")

    resultados_eval_din.append({
        'Usuario': row['nombre'],
        'Recall': recall,
        'Precision': precision,
        'F1': f1,
        'Películas Malas': len(peliculas_malas),
        'Dirección': params['direccion'],
        'Pesos': str(params['weights'])
    })

print(f"\n\n{'='*70}")
print("RESUMEN DE EVALUACIÓN — DINÁMICO")
print(f"{'='*70}")
df_eval_din = pd.DataFrame(resultados_eval_din)
df_eval_din.to_csv('evaluacion_dinamico.csv', index=False)
print(df_eval_din.to_string(index=False))
print(f"\nPromedios:")
print(f"  Recall:    {df_eval_din['Recall'].mean():.1%}")
print(f"  Precision: {df_eval_din['Precision'].mean():.1%}")
print(f"  F1-Score:  {df_eval_din['F1'].mean():.1%}")


## AGREGAR CONCLUSIONES